In [ ]:
import pandas as pd

df = pd.read_csv('ex3.csv')
print('ex3.csv=\n',df.head(3))
print('列ラベル：',df.columns)
print('行列サイズ',df.shape)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
# features = ['x0', 'x1', 'x2', 'x3', 'target']
features = ['x0', 'x1', 'x2', 'x3']
# df_melt = df.melt(id_vars='target', value_vars=features, var_name='feature', value_name='value')
df_melt = df.melt(value_vars=features, var_name='feature', value_name='value')

# 箱ひげ図を描画
plt.figure(figsize=(10,6))
# sns.boxplot(data=df_melt, x='feature', y='value', hue='target')
sns.boxplot(data=df_melt, x='feature', y='value')
plt.title('Dataset Boxplot')
plt.xlabel('Feature')
plt.ylabel('Value')
plt.legend(title='target')
plt.show()
plt.close

In [ ]:
print('平均値：\n',df.mean()[0:5])
print('中央値：\n',df.median()[0:5])
print('標準偏差：\n',df.std()[0:5])

In [ ]:
#print('欠損値の確認')
#print(df.isnull())
print('\nanyメソッドで列単位で欠損値の有無を確認')
print(df.isnull().any(axis=0))
print('\n各列の欠損値の数を調べる')
print(df.isnull().sum())

In [ ]:
# 欠損値をその列の中央値で穴埋め
df2 = df.fillna(df.median())
print('\n各列の欠損値の数を調べる')
print(df2.isnull().sum())

In [ ]:
# 散布図
for name in df.columns:
    if name == 'target':
        continue
    df.plot(kind='scatter', x = name, y = 'target')

In [ ]:
# 外れ値を削除する
no = df2[ ( df2['x2'] < -2 ) & ( df2['target'] > 100 ) ].index
print('Index for Remove=',no)
print( df2[ ( df2['x2'] < -2 ) & ( df2['target'] > 100 ) ])

df3 = df2.drop(no, axis=0)
df3.shape

In [ ]:
# 散布図
for name in df3.columns:
    if name == 'target':
        continue
    df3.plot(kind='scatter', x = name, y = 'target')

In [ ]:
print('平均値：\n',df3.mean()[0:5])
print('中央値：\n',df3.median()[0:5])
print('標準偏差：\n',df3.std()[0:5])

In [ ]:
# データフレームから特徴量と正解データを取り出す
# col = df.columns[0:4]; print('col=',col)
# x = df3[col]
x = df3.loc[: , df.columns[0:4]]
t = df3[df.columns[4]]
print('x=\n',x.head(3),'\nt=\n',t.head(3))

In [ ]:
# ホールドアウト法により訓練データとテストデータに分割する
from sklearn.model_selection import train_test_split

# x_train, y_train：学習に利用する訓練データ
# x_test, y_test：検証に利用するテストデータ
# test_size　: 検証に利用する割合、注意：整数値を指定するとテストデータ件数とみなされる
x_train, x_test, y_train, y_test = train_test_split(x,t,test_size=0.2, random_state=1)
print("学習用訓練データの特徴量の行列サイズ：",x_train.shape)
print(x_train.head(3), "\n")
print("検証用テストデータの特徴量の行列サイズ：",x_test.shape)
print(x_test.head(3),"\n")
print("学習用訓練データの正解の行列サイズ：",y_train.shape)
print(y_train.head(3), "\n")
print("検証用テストデータの正解の行列サイズ：",y_test.shape)
print(y_test.head(3),"\n")

In [ ]:
# 重回帰モデルのLinearRegression関数をインポートする
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train,y_train)

In [ ]:
# モデルのscoreを計算(決定係数R2を計算)
model.score(x_test,y_test)

In [ ]:
# 作成したモデル(平均-標準偏差、平均、平均+標準偏差)で予測を行う
news = [ 
    [df3.mean().iloc[0]-df3.std().iloc[0],df3.mean().iloc[1]-df3.std().iloc[1],df3.mean().iloc[2]-df3.std().iloc[2],df3.mean().iloc[3]-df3.std().iloc[3]],
    [df3.mean().iloc[0],df3.mean().iloc[1],df3.mean().iloc[2],df3.mean().iloc[3]],
    [df3.mean().iloc[0]+df3.std().iloc[0],df3.mean().iloc[1]+df3.std().iloc[1], df3.mean().iloc[2]+df3.std().iloc[2],df3.mean().iloc[3]+df3.std().iloc[3]]
]

for i, new in enumerate(news):
    print(f'Data{i}={new}\n')

news_df = pd.DataFrame(news,columns=x_train.columns)
news_predict = model.predict(news_df)

for i, new_predict in enumerate(news_predict):
    print(f'Predict for Data{i}={new_predict}\n')